# AI-Driven Phishing Email Detection Using NLP
## Final Year Project — Complete Walkthrough

This notebook walks through the full pipeline: loading the dataset, cleaning text,
engineering features, training classifiers, evaluating performance, and inspecting
model explainability. All artifacts (models, vectorizers, plots) are pre-built and
loaded from disk — no need to re-train unless you want to.

## 1. Setup & Imports

In [ ]:
import sys, os, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import config as cfg
from src import data_loader, preprocessing, feature_engineering as fe
from src.explain import PhishingExplainer

warnings.filterwarnings("ignore")
print(f"Project root: {ROOT}")
print("Ready to go!")


## 2. Data Loading

The Kaggle Phishing Email Dataset (`Phishing_Email.csv`) is loaded.
It contains ~18,650 email bodies labelled as "Phishing Email" or "Safe Email".
We drop rows with empty bodies and remove duplicate texts.

In [ ]:
df_raw = data_loader.load_raw()
print(f"\nFinal shape: {df_raw.shape}")
print("\nClass distribution:")
print(df_raw["Email Type"].value_counts().to_string())
df_raw.head(3)


## 3. Text Preprocessing

Each email goes through:
- **HTML stripping** — remove `<...>` tags, unescape `&amp;` / `&nbsp;`
- **URL extraction** — every URL-like token is captured and stored separately
- **Sender domain extraction** — best-effort `From:` line parsing
- **URL & email removal** — blank out actual addresses so they don't pollute the vocabulary
- **Tokenisation & POS-aware lemmatisation** — lowercased, punctuation stripped,
  each word lemmatised using its real part of speech (noun/verb/adjective/adverb)

The result is a cleaned DataFrame with raw text, clean text, tokens, URLs,
and engineered metadata.

In [ ]:
# Load pre-cleaned data (or re-process from scratch)
clean_pkl = cfg.CLEAN_CSV.with_suffix(".pkl")
if clean_pkl.exists():
    with open(clean_pkl, "rb") as fh:
        df_clean = pickle.load(fh)
    print(f"Loaded cleaned data: {len(df_clean)} rows")
else:
    print("Running preprocessing from scratch...")
    df_clean = preprocessing.preprocess_frame(df_raw)
    preprocessing.save_artifacts(df_clean)

df_clean[["raw_length", "sender_domain", "urls", "clean_text"]].head(3)


## 4. Feature Engineering

Three complementary feature representations are used:

1. **TF-IDF** (15,000 unigrams + bigrams) — weights terms by their importance across the corpus
2. **CountVectorizer** (15,000 unigrams + bigrams) — raw term frequencies, used by Naive Bayes
3. **Word2Vec** (100-dimensional embeddings) — trained on the corpus, captures semantic similarity

Additionally, **20+ hand-crafted metadata features** are engineered from domain knowledge:
- URL count, exclamation/question mark counts, uppercase ratio
- Generic greeting detection ("Dear customer/user")
- Sender domain trust score, typosquatting signal
- Suspicious keyword counts, urgency phrase counts
- Currency symbol presence, HTML tag count

A Truncated SVD reduces the sparse TF-IDF matrix to 120 dense components for the neural network.

In [ ]:
# Load the pre-built feature bundle
bundle = fe.load_bundle()
print(f"TF-IDF vocabulary size: {len(bundle.tfidf.vocabulary_)}")
print(f"CountVectorizer vocab size: {len(bundle.count.vocabulary_)}")
print(f"SVD components: {bundle.svd.n_components}")
print(f"Word2Vec vocab: {len(bundle.w2v.wv)}")
print(f"Metadata columns: {len(bundle.meta_columns)}")


## 5. Model Training & Evaluation

Four classifiers are trained and tuned:

| Model | Input features | Hyperparameter tuning |
|---|---|---|
| Logistic Regression | TF-IDF + metadata | Grid search: C, class_weight |
| Multinomial Naive Bayes | TF-IDF only | Grid search: alpha, fit_prior |
| Random Forest | TF-IDF + metadata | Randomized search: n_estimators, max_depth |
| MLP (Neural Network) | SVD(120) + metadata | Randomized search: layers, alpha, lr |

A Word2Vec + Logistic Regression baseline is also fitted.

All models are evaluated on a held-out 20% test set, and 5-fold
cross-validation F1 scores are reported.

In [ ]:
# Load evaluation results
with open(cfg.MODEL_DIR / "metrics.json") as fh:
    metrics = json.load(fh)
with open(cfg.MODEL_DIR / "cv_results.json") as fh:
    cv = json.load(fh)
with open(cfg.MODEL_DIR / "timings.json") as fh:
    timings = json.load(fh)

# Build comparison table
rows = []
pretty = {
    "logistic_regression": "Logistic Regression",
    "naive_bayes": "Naive Bayes",
    "random_forest": "Random Forest",
    "neural_network": "Neural Network",
    "word2vec_lr": "Word2Vec + LR",
}
for name, m in metrics.items():
    cv_info = cv.get(name, {})
    tim = timings.get(name, {})
    rows.append({
        "Model": pretty.get(name, name),
        "Accuracy": f"{m['accuracy']:.4f}",
        "Precision": f"{m['precision']:.4f}",
        "Recall": f"{m['recall']:.4f}",
        "F1 Score": f"{m['f1']:.4f}",
        "ROC AUC": f"{m['roc_auc']:.4f}",
        "CV F1 (mean+/-std)": (f"{cv_info.get('mean', '-'):.4f} +/- "
                              f"{cv_info.get('std', '-'):.4f}"),
        "Train time (s)": f"{tim.get('train_seconds', '-'):.2f}",
    })

table = pd.DataFrame(rows)
display(table)

best = max(metrics, key=lambda k: metrics[k]["f1"])
print(f"\nBest model: {pretty[best]} (F1 = {metrics[best]['f1']:.4f})")


## 6. Visualisations

All plots are pre-generated by the evaluation pipeline and stored in `visualizations/`.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

plot_files = [
    ("class_distribution.png", "Class Distribution"),
    ("metric_bars.png", "Model Metrics"),
    ("roc_curves.png", "ROC Curves"),
    ("feature_importance.png", "Feature Importance"),
    ("cv_boxplot.png", "CV F1 Scores"),
    ("timings.png", "Training Times"),
]

for ax, (fname, title) in zip(axes.flat, plot_files):
    path = cfg.VIZ_DIR / fname
    if path.exists():
        img = plt.imread(path)
        ax.imshow(img)
        ax.set_title(title, fontsize=12, fontweight="bold")
        ax.axis("off")
    else:
        ax.text(0.5, 0.5, f"Missing: {fname}", ha="center", va="center")
        ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
model_names = ["logistic_regression", "naive_bayes", "random_forest", "neural_network"]
pretty_names = ["Logistic Regression", "Naive Bayes", "Random Forest", "Neural Network"]

for ax, name, pname in zip(axes, model_names, pretty_names):
    path = cfg.VIZ_DIR / f"cm_{name}.png"
    if path.exists():
        img = plt.imread(path)
        ax.imshow(img)
        ax.set_title(pname, fontsize=11, fontweight="bold")
        ax.axis("off")

plt.tight_layout()
plt.show()


## 7. Explainability Demo

The `PhishingExplainer` class runs an email through the full pipeline
and produces a structured explanation with risk score, URL verdicts,
sender-domain checks, and plain-English reasons.

In [ ]:
explainer = PhishingExplainer("logistic_regression")

phish_email = (
    "From: PayPaI Security <security@paypa1-verify.com>\n"
    "Subject: Your account has been limited\n\n"
    "Dear valued customer,\n"
    "We have detected unusual activity on your account. "
    "Your account has been temporarily restricted.\n\n"
    "To restore your account, please verify your information immediately:\n"
    "http://185.220.101.44/login/paypal/verify-account.php\n\n"
    "If you do not verify within 24 hours, your account will be "
    "permanently suspended.\n\n"
    "Thank you,\nPayPal Security Team"
)

result = explainer.explain_email(phish_email)
print(f"Label: {result.label}")
print(f"Risk: {result.risk_score}/100 ({result.risk_level})")
print(f"Phishing probability: {result.phishing_probability:.1%}")
print()
print("Suspicious words:", result.suspicious_words[:8], "...")
print()
for v in result.url_verdicts:
    print(f"  {v.label}: {v.url[:60]}...")
    for f in v.flags:
        print(f"    - {f}")
print()
print("Why this verdict?")
for i, r in enumerate(result.reasons, 1):
    print(f"  {i}. {r}")


In [ ]:
safe_email = (
    "From: alice@company.com\n"
    "Subject: Meeting tomorrow\n\n"
    "Hi team,\n"
    "Just a reminder about the sprint retro tomorrow at 10 AM.\n"
    "Best,\nAlice"
)

result_safe = explainer.explain_email(safe_email)
print(f"Label: {result_safe.label}")
print(f"Risk: {result_safe.risk_score}/100 ({result_safe.risk_level})")
print(f"Phishing probability: {result_safe.phishing_probability:.1%}")
print()
print("Why this verdict?")
for i, r in enumerate(result_safe.reasons, 1):
    print(f"  {i}. {r}")


## 8. Conclusions

### Key Findings

- **Logistic Regression** achieves the best F1 score (0.9754) with training
  under 10 seconds and inference at 4 us/email — the best accuracy-speed trade-off.
- **All four classifiers** perform above 0.97 F1, showing that classical NLP
  pipelines are still highly effective for text-based phishing detection.
- **TF-IDF + metadata** consistently outperforms Word2Vec embeddings by ~2 F1 points,
  likely because phishing language is lexically distinctive (specific keywords matter
  more than semantic similarity).
- **Hand-crafted features** (URL analysis, sender reputation, urgency detection)
  add ~0.3-0.5 F1 across all models compared to using TF-IDF alone.
- **Explainability** is crucial: the risk-score breakdown and per-reason verdicts
  make the system auditable and trustworthy.

### Limitations

- The dataset contains only email bodies, not full RFC-822 messages (~89% lack sender headers).
- Phishing language evolves; periodic retraining is essential.
- The system can be evaded by sophisticated attackers aware of the feature representations.

### Future Work

- Integrate full email headers (SPF, DKIM, DMARC checks)
- Deep learning with BERT/RoBERTa for sequence classification
- Browser extension for real-time client-side scanning
- Multi-language phishing detection